# Terra generation

The model `ldm_v4` was fine-tuned with two conditions, both can be set at generation time:

- **who** (`WHO`): `'terra'`, `'other'` (any other cat) or `None` (the "null" condition);
- **type** (`TYPE`): a DINOv2 feature cluster, `None`, `'terra'` (Terra's main cluster) or a number 0-99.
  The clusters are mostly about coat color and breed. Terra is in the cluster of long-haired colorpoints together with ~330 similar cats.
  The "Cat types" cell shows what each type looks like; all of them at once: `review/clusters_*.jpg` in the dataset folder.

Classifier-free guidance (CFG) is used at generation: the conditioned model output is pushed away from the unconditioned one
with strength `W`. Larger `W` = more similar to Terra, but less diversity and a higher risk of oversaturated colors.
Autoguidance with a weak model (`GUIDE`) can be added on top.

30 photos of Terra are held out and were not used in training (`terra_holdout.json`). The last cell shows,
for each image, the nearest training photo and marks copies.

**Which snapshot to use** (`runs/eval/terra_v4/`, 512 "Terra + type" samples; held-out real photos have 3.3% copies):

| Snapshot | w | Resembles Terra | Copies |
|---|---|---|---|
| 5 000 | 2 | 97.7% | 0.6% |
| **7 500** | **2** | **98.8%** | **2.5%** |
| 7 500 | 3 | 100% | 9.2% |
| 10 000 | 2 | 99.6% | 15.6% |
| 25 000 (`ckpt/last.ckpt`) | 2 | 99.8% | 60.4% |

Beyond 7 500 the model increasingly repeats the training photos, so the default is `ema_0007500.pt`.

Sampler: `METHOD = 'sde'` (stochastic, 250 steps) or `'heun'` (ODE, 50 steps, deterministic: one noise = one image).
On unconditional models Heun gave smeared images; the "SDE and ODE" cell compares them on the same noises.

In [ ]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from diffusers import AutoencoderKL
from PIL import Image, ImageOps

from dataset import read_manifest
from guidance import CFG, Guided
from models import UNet
from train import VPSDE, sample

RUN_DIR = Path('runs/ldm_v4')
DATA = Path.home() / 'DiffusionLearn_data/pretrain_v1'
DEVICE = 'cuda'

cfg = json.loads((RUN_DIR / 'config.json').read_text())
WHO_NULL, K = cfg['cond_sizes']        # "null" values of the conditions: who = 2, type = 100
TERRA_TYPE = cfg['terra_type']         # Terra's main cluster
print(f"steps {cfg['steps']}, Terra {cfg['terra_frac']:.0%} of examples, clusters {K}, Terra type {TERRA_TYPE}, scale {cfg['scale']:.5f}")

vae = AutoencoderKL.from_pretrained(cfg['vae'], local_files_only=True, low_cpu_mem_usage=False)
vae = vae.to(DEVICE).eval().requires_grad_(False)
sde = VPSDE(pred=cfg['pred'])
bf16 = lambda: torch.autocast('cuda', dtype=torch.bfloat16)


def load_ema(path):
    """EMA weights from ckpt/last.ckpt or ema_*.pt of any run, configuration from its config.json. Returns the model and the step."""
    path = Path(path)
    run = path.parent.parent if path.parent.name == 'ckpt' else path.parent
    c = json.loads((run / 'config.json').read_text())
    ck = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    sd = ck['ema'] if 'ema' in ck else {k[4:]: v for k, v in ck['state_dict'].items() if k.startswith('ema.')}
    m = UNet(img_size=64, in_ch=4, base=c['base'], mults=[int(v) for v in c['mults'].split(',')], num_res=c['num_res'],
             attn_res=[int(v) for v in c['attn_res'].split(',')], dropout=c['dropout'], norm_groups=32,
             head_ch=c.get('head_ch') or None, qk_norm=c.get('qk_norm', False), cond_sizes=c.get('cond_sizes', []))
    m.load_state_dict(sd)
    return m.to(DEVICE).eval().requires_grad_(False), ck.get('step', ck.get('global_step'))


@torch.no_grad()
def decode(z, chunk=8):
    """Model latents (B, 4, 64, 64) -> uint8 images (B, 3, 512, 512) on CPU."""
    out = []
    for i in range(0, len(z), chunk):
        x = vae.decode(z[i:i + chunk].to(DEVICE).float() / cfg['scale']).sample
        out.append(((x.clamp(-1, 1) + 1) * 127.5).round().byte().cpu())
    return torch.cat(out)


def to_np(img):
    """(3, H, W) uint8 -> (H, W, 3) for imshow."""
    return img.permute(1, 2, 0).numpy()


@torch.no_grad()
def generate(model, n, seed=0, who='terra', type_=None, w=2.0, method='sde', steps=None, guide=None, w_ag=1.5, chunk=16):
    """
    n images with conditions. Returns latents (n, 4, 64, 64) on CPU.
      who      : 'terra', 'other' (any other cat) or None (the "null" condition)
      type_    : None ("null"), 'terra' (Terra's main cluster) or cluster number 0..K-1
      w        : CFG strength relative to the unconditioned output, 1 = no CFG. If both conditions are null, w has no effect
      method   : 'sde' (Euler-Maruyama) or 'heun' (ODE)
      steps    : default 250 for SDE and 50 for Heun
      guide    : weak model for autoguidance with the same conditions, w_ag is its strength
    The same seed gives the same starting noises under any settings, so variants can be compared.
    """
    steps = steps or (250 if method == 'sde' else 50)
    y = torch.tensor([{'terra': 1, 'other': 0, None: WHO_NULL}[who],
                      K if type_ is None else TERRA_TYPE if type_ == 'terra' else int(type_)])
    y0 = torch.tensor([WHO_NULL, K])
    net = CFG(model, y, y0, w)
    if guide is not None:
        net = Guided(net, CFG(guide, y, y0, w), w_ag)
    x0 = torch.randn(n, 4, 64, 64, generator=torch.Generator().manual_seed(seed))
    out = []
    for i in range(0, n, chunk):
        torch.manual_seed(seed * 1_000_003 + i)                     # the SDE noise is repeated too
        out.append(sample(net, sde, x0[i:i + chunk].to(DEVICE), steps=steps, rho=1.0 if method == 'sde' else 2.0,
                          method=method, amp_ctx=bf16, clip=False).cpu())
    return torch.cat(out)


def show(rows, labels, title, ncol):
    """Image grid: rows = list of uint8 batches, row = variant."""
    fig, axs = plt.subplots(len(rows), ncol, figsize=(2.8 * ncol, 2.9 * len(rows)), squeeze=False)
    for r, ims in enumerate(rows):
        for c in range(ncol):
            axs[r, c].imshow(to_np(ims[c]))
            axs[r, c].set_xticks([])
            axs[r, c].set_yticks([])
        axs[r, 0].set_ylabel(labels[r], fontsize=12)
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

## Cat types

Random photos of other cats from one cluster: this shows what a `TYPE` number means.

In [ ]:
SHOW_TYPE = TERRA_TYPE       # cluster number 0..99

assign = json.loads((DATA / 'clusters.json').read_text())['assign']
members = [p for p, c in assign.items() if c == SHOW_TYPE and not p.startswith('terra/')]
pick = random.Random(0).sample(members, min(8, len(members)))
fig, axs = plt.subplots(1, len(pick), figsize=(2.6 * len(pick), 2.9), squeeze=False)
for ax, p in zip(axs[0], pick):
    ax.imshow(ImageOps.fit(ImageOps.exif_transpose(Image.open(DATA / p)).convert('RGB'), (256, 256)))
    ax.axis('off')
plt.suptitle(f'type {SHOW_TYPE}: {len(members)} frames of other cats' + (', Terra main type' if SHOW_TYPE == TERRA_TYPE else ''))
plt.tight_layout()
plt.show()

## Generation

Images are saved to `runs/ldm_v4/generated_terra/<name>/`, the name is built from the parameters.

In [ ]:
# ------------------------------------------------------------- parameters
CKPT = 'ema_0007500.pt'      # best balance of resemblance and novelty; 'ckpt/last.ckpt' (25 000) mostly copies
WHO = 'terra'                # 'terra', 'other' or None
TYPE = None                  # None, 'terra' (Terra's main cluster) or cluster number 0..99
W = 2.0                      # CFG strength: 1 = none, 2 is best by the metrics, 3 copies more often
METHOD = 'sde'               # 'sde' or 'heun' (ODE)
STEPS = None                 # None = 250 for SDE, 50 for Heun
# GUIDE = None                 # autoguidance with a weak model
GUIDE = 'runs/ldm_v3/ckpt/last.ckpt'   # ldm_v3: the same model without conditions and without Terra
W_AG = 1.5
SEED = 1
BATCH = 16
COLS = 4

# ------------------------------------------------------------- generation
model, step = load_ema(RUN_DIR / CKPT)
guide = load_ema(GUIDE)[0] if GUIDE else None
lat = generate(model, BATCH, seed=SEED, who=WHO, type_=TYPE, w=W, method=METHOD, steps=STEPS, guide=guide, w_ag=W_AG)
imgs = decode(lat)
del guide
torch.cuda.empty_cache()

name = (f'step{step:07d}_{WHO or "any"}' + (f'_type{TYPE}' if TYPE is not None else '') + f'_w{W}_{METHOD}{STEPS or ""}'
        + (f'_ag-{Path(GUIDE).parts[-3]}-w{W_AG}' if GUIDE else '') + f'_seed{SEED}')
out_dir = RUN_DIR / 'generated_terra' / name
out_dir.mkdir(parents=True, exist_ok=True)
for i, im in enumerate(imgs):
    Image.fromarray(to_np(im)).save(out_dir / f'{i:03d}.png')
print(f'{BATCH} images saved to {out_dir}')

rows = math.ceil(BATCH / COLS)
fig, axs = plt.subplots(rows, COLS, figsize=(4.5 * COLS, 4.7 * rows), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for i, im in enumerate(imgs):
    axs.flat[i].imshow(to_np(im))
    axs.flat[i].set_title(str(i))
plt.suptitle(name, fontsize=15)
plt.tight_layout()
plt.show()

## CFG strength

The same noises and conditions as in the generation cell, row = `W`. Shows how resemblance grows and diversity drops.

In [ ]:
W_LIST = [1.0, 1.5, 2.0, 3.0, 4.0]
N_CMP = 6

rows = [decode(generate(model, N_CMP, seed=SEED, who=WHO, type_=TYPE, w=w, method=METHOD, steps=STEPS)) for w in W_LIST]
show(rows, [f'W = {w}' for w in W_LIST], f'step {step}, who {WHO}, type {TYPE}, {METHOD}', N_CMP)

## SDE and ODE

The same starting noises and conditions. SDE adds fresh noise at every step and thereby corrects the model's errors,
ODE (Heun) runs deterministically and usually gives smoother but also blurrier images.

In [ ]:
N_CMP = 6
VARIANTS = [('sde', 250), ('sde', 1000), ('heun', 50), ('heun', 200)]

rows = [decode(generate(model, N_CMP, seed=SEED, who=WHO, type_=TYPE, w=W, method=m, steps=st)) for m, st in VARIANTS]
show(rows, [f'{m} {st}' for m, st in VARIANTS], f'step {step}, who {WHO}, type {TYPE}, W {W}', N_CMP)

## Is this Terra new or a copy

As in `tools/eval_terra.py`: each Terra training photo is split into crops by the training rule
(box inside, several sizes and shifts, flip) at 64x64, and the nearest crop is searched for each image.
d1 = distance to the nearest photo, d2 = to the second nearest other photo, `ratio = d1 / d2`.
A copy is much closer to one photo than to the rest: `ratio < 0.7`, the caption is red.
For the 30 held-out real photos of Terra this happens about 3% of the time.

The first run of the cell takes about a minute: crops of the 254 training photos are built.

In [ ]:
from tools.eval_terra import copy_ratio, copy_refs

N_SHOW = 8

if 'refs' not in globals():
    hold = set(json.loads((DATA / 'terra_holdout.json').read_text()))
    train_rows = [r for r in read_manifest(DATA / 'manifest.jsonl') if r['source'] == 'terra' and r['path'] not in hold]
    refs, owner = copy_refs(DATA, train_rows)

x = imgs[:N_SHOW].float() / 127.5 - 1
ratio, nn = copy_ratio(x, refs, owner, len(train_rows))
ratio, nn = ratio.cpu(), nn.cpu()
print(f'ratio per image: ' + ', '.join(f'{v:.2f}' for v in ratio.tolist())
      + f'; copies {(ratio < 0.7).sum().item()} of {N_SHOW}')

fig, axs = plt.subplots(math.ceil(N_SHOW / 2), 4, figsize=(16, 4.2 * math.ceil(N_SHOW / 2)), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for j in range(N_SHOW):
    r, c = divmod(j, 2)
    real = ImageOps.fit(ImageOps.exif_transpose(Image.open(DATA / train_rows[nn[j]]['path'])).convert('RGB'), (512, 512))
    axs[r, 2 * c].imshow(to_np(imgs[j]))
    axs[r, 2 * c].set_title(f'{j}: ratio {ratio[j]:.2f}' + (', copy' if ratio[j] < 0.7 else ''),
                            color='red' if ratio[j] < 0.7 else 'black')
    axs[r, 2 * c + 1].imshow(real)
    axs[r, 2 * c + 1].set_title('nearest training photo')
plt.tight_layout()
plt.show()